# Pull GEDI L1B and L2A footprints within a raster bounding box

This notebook uses a local raster template to define the area of interest, transforms the raster bounds to WGS84 for NASA Earthdata/CMR search, pulls both GEDI L1B (`GEDI01_B`) and GEDI L2A (`GEDI02_A`) granules, extracts footprint-level records, clips them to the raster extent, and writes spatial/tabular outputs.

Default input raster:

```text
C:\NCA_DATA\templates\grid_template_10m_26911_uint8.tif
```

Key outputs:

- L1B footprint index table/spatial layer inside the raster bbox
- L2A footprint metrics table/spatial layer inside the raster bbox
- Joined L1B/L2A footprint table using `shot_number` where both products overlap

Important: the L1B product contains geolocated waveform data. This notebook does **not** flatten full waveform arrays into the main footprint table by default. Instead, it stores per-shot waveform index fields where available, plus a helper for extracting a selected shot waveform later. That keeps the output table manageable.


## 0. Install dependencies if needed

Run this cell once in the conda environment if any imports fail.


In [ ]:
# Uncomment if needed
# %pip install earthaccess rasterio geopandas shapely pyproj h5py pandas numpy tqdm pyogrio


## 1. Imports and configuration


In [ ]:
from pathlib import Path
import os
import re
import warnings

import earthaccess
import h5py
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import transform_bounds
from rasterio.transform import rowcol
from shapely.geometry import box
from tqdm.std import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

# -------------------------
# User settings
# -------------------------
RASTER_PATH = Path(r"A:\NCA_DATA\templates\grid_template_10m_26911_uint8.tif")

# GEDI products requested
GEDI_PRODUCTS = {
    "L1B": "GEDI01_B",   # Level 1B geolocated waveform data
    "L2A": "GEDI02_A",   # Level 2A elevation and RH height metrics
}
GEDI_VERSION = "002"

# GEDI V002 public archive. Narrow this if you only need a specific season/year.
TEMPORAL_RANGE = ("2019-04-18", "2024-11-29")

# Increase if your AOI/date range returns many granules.
MAX_GRANULES_PER_PRODUCT = 500

# True = download HDF5 files locally; False = stream/open with earthaccess.
# For repeated development, downloading is usually more stable.
DOWNLOAD_GRANULES = True

OUT_DIR = Path(r"A:\NCA_DATA\GEDI\NCA_template_bbox_L1B_L2A")
OUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR = OUT_DIR / "raw_h5"
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "GEDI_L1B_L2A_footprints_in_template_bbox.gpkg"
OUT_L1B_CSV = OUT_DIR / "GEDI01_B_L1B_footprints_in_template_bbox.csv"
OUT_L2A_CSV = OUT_DIR / "GEDI02_A_L2A_footprints_in_template_bbox.csv"
OUT_JOINED_CSV = OUT_DIR / "GEDI01_B_GEDI02_A_joined_by_shot_number_in_template_bbox.csv"

print("Output directory:", OUT_DIR)


: 

## 2. Read raster bounds and transform to WGS84

The GEDI/CMR search needs longitude/latitude coordinates. The raster can stay in its native CRS for exact spatial filtering and grid row/column indexing.


In [ ]:
if not RASTER_PATH.exists():
    raise FileNotFoundError(f"Raster not found: {RASTER_PATH}")

with rasterio.open(RASTER_PATH) as src:
    raster_crs = src.crs
    raster_bounds = src.bounds
    raster_transform = src.transform
    raster_width = src.width
    raster_height = src.height
    raster_res = src.res

print("Raster CRS:", raster_crs)
print("Raster bounds in raster CRS:", raster_bounds)
print("Raster size:", raster_width, "x", raster_height)
print("Raster resolution:", raster_res)

west, south, east, north = transform_bounds(
    raster_crs,
    "EPSG:4326",
    raster_bounds.left,
    raster_bounds.bottom,
    raster_bounds.right,
    raster_bounds.top,
    densify_pts=21,
)

cmr_bbox = (west, south, east, north)
print("CMR/WGS84 bounding box:", cmr_bbox)

raster_bbox_poly = box(raster_bounds.left, raster_bounds.bottom, raster_bounds.right, raster_bounds.top)
aoi_native = gpd.GeoDataFrame({"name": ["raster_bbox"]}, geometry=[raster_bbox_poly], crs=raster_crs)
aoi_wgs84 = aoi_native.to_crs("EPSG:4326")


## 3. Authenticate with NASA Earthdata


In [ ]:
# This uses an existing .netrc if present, or prompts for NASA Earthdata credentials.
auth = earthaccess.login(persist=True)
auth


## 4. Search and prepare L1B and L2A granules

CMR returns candidate granules that intersect the raster-derived WGS84 bbox. Exact footprint clipping happens after reading each HDF5 file.


In [ ]:
def get_granule_name(granule):
    """
    Extract a readable granule name from an earthaccess DataGranule object.

    Different earthaccess versions expose metadata differently, so this avoids
    using granule.summary(), which is not available in your installed version.
    """
    try:
        return granule["umm"].get("GranuleUR", "unknown granule")
    except Exception:
        pass

    try:
        return granule.get("umm", {}).get("GranuleUR", "unknown granule")
    except Exception:
        pass

    try:
        return granule.get("meta", {}).get("native-id", "unknown granule")
    except Exception:
        pass

    return "unknown granule"


def search_gedi_product(short_name, label):
    results = earthaccess.search_data(
        short_name=short_name,
        version=GEDI_VERSION,
        bounding_box=cmr_bbox,
        temporal=TEMPORAL_RANGE,
        count=MAX_GRANULES_PER_PRODUCT,
    )

    print(f"{label} / {short_name}: found {len(results)} candidate granules")

    if results:
        print("Example granules:")
        for i, granule in enumerate(results[:3]):
            print(f"  [{i}] {get_granule_name(granule)}")

    return results


results_by_level = {
    level: search_gedi_product(short_name, level)
    for level, short_name in GEDI_PRODUCTS.items()
}

if not results_by_level["L1B"]:
    raise RuntimeError(
        "No GEDI L1B granules found. Check bbox, dates, product version, or MAX_GRANULES_PER_PRODUCT."
    )

if not results_by_level["L2A"]:
    raise RuntimeError(
        "No GEDI L2A granules found. Check bbox, dates, product version, or MAX_GRANULES_PER_PRODUCT."
    )

In [ ]:
def gedi_pair_key(granule_name):
    """
    Extract stable orbit/time/track key from GEDI granule name.

    Example:
    GEDI01_B_2019109210809_O01988_03_T02056_02_005_01_V002
    -> 2019109210809_O01988_03_T02056_02
    """
    parts = granule_name.split("_")

    # parts:
    # [0] GEDI01
    # [1] B or A
    # [2] timestamp
    # [3] orbit
    # [4] sub-orbit/beam group
    # [5] track
    # [6] processing segment
    return "_".join(parts[2:7])


granule_tables = {}

for level, granules in results_by_level.items():
    rows = []
    for g in granules:
        name = get_granule_name(g)
        rows.append(
            {
                "level": level,
                "granule_name": name,
                "pair_key": gedi_pair_key(name),
            }
        )
    granule_tables[level] = pd.DataFrame(rows)

l1b_keys = set(granule_tables["L1B"]["pair_key"])
l2a_keys = set(granule_tables["L2A"]["pair_key"])

print("L1B granules:", len(l1b_keys))
print("L2A granules:", len(l2a_keys))
print("Matched granule keys:", len(l1b_keys & l2a_keys))
print("L1B only:", len(l1b_keys - l2a_keys))
print("L2A only:", len(l2a_keys - l1b_keys))

display(granule_tables["L1B"].head())
display(granule_tables["L2A"].head())

## 5. Download or open HDF5 sources


In [ ]:
def prepare_h5_sources(results, short_name):
    product_raw_dir = RAW_DIR / short_name
    product_raw_dir.mkdir(parents=True, exist_ok=True)
    if DOWNLOAD_GRANULES:
        paths = earthaccess.download(results, local_path=str(product_raw_dir))
        return [Path(p) for p in paths]
    return earthaccess.open(results)

h5_sources_by_level = {}
for level, short_name in GEDI_PRODUCTS.items():
    h5_sources_by_level[level] = prepare_h5_sources(results_by_level[level], short_name)
    print(f"{level}: prepared {len(h5_sources_by_level[level])} HDF5 sources")

h5_sources_by_level["L1B"][:2], h5_sources_by_level["L2A"][:2]


## 6. HDF5 extraction helpers

These helpers tolerate the main coordinate-path differences between L1B and L2A. They extract a compact footprint table and avoid expanding the full L1B waveform arrays unless requested later.


In [ ]:
BEAM_RE = re.compile(r"^BEAM\d{4}$")


def list_beams(h5):
    return sorted([k for k in h5.keys() if BEAM_RE.match(k)])


def dataset_exists(group, path):
    try:
        obj = group[path]
        return isinstance(obj, h5py.Dataset)
    except Exception:
        return False


def read_first_dataset(group, candidate_paths):
    for path in candidate_paths:
        if dataset_exists(group, path):
            return np.asarray(group[path]), path
    return None, None


def to_1d(arr, n=None):
    if arr is None:
        return None
    arr = np.asarray(arr)
    if arr.ndim == 0:
        return None
    if arr.ndim > 1:
        return None
    if n is not None and len(arr) != n:
        return None
    if arr.dtype.kind in {"S", "O"}:
        return arr.astype(str)
    return arr


def read_optional_1d(group, candidate_paths, n):
    arr, used_path = read_first_dataset(group, candidate_paths)
    arr = to_1d(arr, n=n)
    return arr, used_path


def add_optional_column(out, col_name, group, candidate_paths, n):
    arr, used_path = read_optional_1d(group, candidate_paths, n)
    if arr is not None:
        out[col_name] = arr
        return used_path
    return None


def infer_granule_name(h5_source):
    try:
        return Path(str(h5_source)).name
    except Exception:
        return str(h5_source)


def extract_beam_records(h5, beam_name, source_name, level):
    beam = h5[beam_name]

    lat, lat_path = read_first_dataset(beam, [
        "lat_lowestmode",
        "geolocation/lat_lowestmode",
        "geolocation/latitude_lowestmode",
        "geolocation/latitude_bin0",
        "latitude_bin0",
    ])
    lon, lon_path = read_first_dataset(beam, [
        "lon_lowestmode",
        "geolocation/lon_lowestmode",
        "geolocation/longitude_lowestmode",
        "geolocation/longitude_bin0",
        "longitude_bin0",
    ])

    lat = to_1d(lat)
    lon = to_1d(lon)
    if lat is None or lon is None or len(lat) != len(lon):
        return pd.DataFrame()

    n = len(lat)
    out = {
        "level": np.repeat(level, n),
        "source_granule": np.repeat(source_name, n),
        "beam": np.repeat(beam_name, n),
        "lat": lat,
        "lon": lon,
    }

    # Shot number is the safest key for joining L1B and L2A.
    shot, shot_path = read_optional_1d(beam, ["shot_number", "geolocation/shot_number"], n)
    if shot is not None:
        # Keep as string to avoid precision loss in CSV/parquet workflows.
        out["shot_number"] = shot.astype(str)

    # Common-ish beam/product metadata.
    common_fields = {
        "beam_type": ["beam_type", "geolocation/beam_type"],
        "channel": ["channel", "geolocation/channel"],
        "degrade_flag": ["degrade_flag", "geolocation/degrade_flag"],
        "stale_return_flag": ["stale_return_flag"],
        "landsat_water_persistence": ["geolocation/landsat_water_persistence"],
        "digital_elevation_model": ["digital_elevation_model", "geolocation/digital_elevation_model"],
        "solar_elevation": ["solar_elevation", "geolocation/solar_elevation"],
        "sensitivity": ["sensitivity"],
    }
    used_paths = {"lat": lat_path, "lon": lon_path, "shot_number": shot_path}
    for col, paths in common_fields.items():
        used = add_optional_column(out, col, beam, paths, n)
        if used:
            used_paths[col] = used

    if level == "L1B":
        l1b_fields = {
            "rx_sample_start_index": ["rx_sample_start_index"],
            "rx_sample_count": ["rx_sample_count"],
            "tx_sample_start_index": ["tx_sample_start_index"],
            "tx_sample_count": ["tx_sample_count"],
            "rx_energy": ["rx_energy"],
            "tx_energy": ["tx_energy"],
            "rx_maxamp": ["rx_maxamp"],
            "rx_mean": ["rx_mean"],
            "rx_sd": ["rx_sd"],
            "geolocation_quality_flag": ["geolocation/quality_flag"],
            "elevation_bin0": ["geolocation/elevation_bin0", "elevation_bin0"],
        }
        for col, paths in l1b_fields.items():
            used = add_optional_column(out, col, beam, paths, n)
            if used:
                used_paths[col] = used

    if level == "L2A":
        l2a_fields = {
            "quality_flag": ["quality_flag"],
            "num_detectedmodes": ["num_detectedmodes"],
            "selected_algorithm": ["selected_algorithm"],
            "elev_lowestmode": ["elev_lowestmode"],
            "elev_highestreturn": ["elev_highestreturn"],
            "elev_ground": ["elev_lowestmode"],
            "delta_time": ["delta_time"],
            "surface_flag": ["surface_flag"],
        }
        for col, paths in l2a_fields.items():
            used = add_optional_column(out, col, beam, paths, n)
            if used:
                used_paths[col] = used

        # RH is commonly an n x 101 array. Keep selected RH metrics rather than all 101 by default.
        if dataset_exists(beam, "rh"):
            rh = np.asarray(beam["rh"])
            if rh.ndim == 2 and rh.shape[0] == n:
                for idx in [0, 5, 10, 25, 50, 75, 90, 95, 98, 100]:
                    if idx < rh.shape[1]:
                        out[f"rh{idx}"] = rh[:, idx]
                used_paths["rh_selected"] = "rh"

    df = pd.DataFrame(out)
    df.attrs["used_paths"] = used_paths
    return df


def extract_gedi_records(h5_source, level):
    source_name = infer_granule_name(h5_source)
    dfs = []
    with h5py.File(h5_source, "r") as h5:
        for beam_name in list_beams(h5):
            df = extract_beam_records(h5, beam_name, source_name, level)
            if not df.empty:
                dfs.append(df)
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


## 7. Extract footprint records from L1B and L2A


In [ ]:
# =============================================================================
# Rebuild local GEDI HDF5 source lists after kernel restart
# =============================================================================

from pathlib import Path

RAW_H5_ROOT = Path(r"A:\NCA_DATA\GEDI\NCA_template_bbox_L1B_L2A\raw_h5")

l1b_dir = RAW_H5_ROOT / "GEDI01_B"
l2a_dir = RAW_H5_ROOT / "GEDI02_A"

if not l1b_dir.exists():
    raise FileNotFoundError(f"L1B directory not found: {l1b_dir}")

if not l2a_dir.exists():
    raise FileNotFoundError(f"L2A directory not found: {l2a_dir}")

h5_sources_by_level = {
    "L1B": sorted(l1b_dir.glob("*.h5")),
    "L2A": sorted(l2a_dir.glob("*.h5")),
}

print(f"L1B HDF5 files found: {len(h5_sources_by_level['L1B']):,}")
print(f"L2A HDF5 files found: {len(h5_sources_by_level['L2A']):,}")

print("\nExample L1B:")
for p in h5_sources_by_level["L1B"][:3]:
    print(" ", p.name)

print("\nExample L2A:")
for p in h5_sources_by_level["L2A"][:3]:
    print(" ", p.name)

In [ ]:
def extract_product_table(h5_sources, level):
    dfs = []
    failed = []
    for h5_source in tqdm(h5_sources, desc=f"Reading {level} HDF5 granules"):
        try:
            df = extract_gedi_records(h5_source, level=level)
            if not df.empty:
                dfs.append(df)
        except Exception as exc:
            failed.append((str(h5_source), repr(exc)))

    if failed:
        print(f"{level}: failed to read {len(failed)} granules")
        for f, e in failed[:10]:
            print(" -", f, e)

    if not dfs:
        print(f"{level}: no footprint records extracted")
        return pd.DataFrame()

    out = pd.concat(dfs, ignore_index=True)
    print(f"{level}: raw extracted footprints = {len(out):,}")
    return out

l1b_raw = extract_product_table(h5_sources_by_level["L1B"][:2], "L1B")
l2a_raw = extract_product_table(h5_sources_by_level["L2A"][:2], "L2A")

print("L1B columns:", sorted(l1b_raw.columns.tolist()))
print("L2A columns:", sorted(l2a_raw.columns.tolist()))


## 8. Spatially filter footprints to the raster extent and add grid row/column


In [ ]:
# =============================================================================
# 7. Extract GEDI footprint records and immediately trim to core AOI
# =============================================================================
#
# Important:
#   Do NOT extract all bbox footprints into one massive table before clipping.
#   This cell reads each HDF5 granule, extracts footprint records, clips that
#   granule to the core AOI, and only retains the AOI-intersecting shots.
#
# Inputs expected from earlier cells:
#   - h5_sources_by_level
#   - raster_crs
#   - extract_gedi_records()
#
# Outputs:
#   - l1b_raw      core-AOI-filtered L1B footprint table
#   - l2a_raw      core-AOI-filtered L2A footprint table
#   - l1b_native   alias for l1b_raw
#   - l2a_native   alias for l2a_raw
#   - l1b_core_gdf alias for l1b_raw
#   - l2a_core_gdf alias for l2a_raw
# =============================================================================

from pathlib import Path
import gc
import pandas as pd
import geopandas as gpd
from tqdm.auto import tqdm

CORE_AOI_PATH = Path(r"A:\NCA_DATA\templates\NCA_NAD83UTM11N.shp")

if not CORE_AOI_PATH.exists():
    raise FileNotFoundError(f"Core AOI shapefile not found: {CORE_AOI_PATH}")

core_aoi = gpd.read_file(CORE_AOI_PATH)

if core_aoi.empty:
    raise ValueError(f"Core AOI shapefile is empty: {CORE_AOI_PATH}")

if core_aoi.crs is None:
    raise ValueError(
        f"Core AOI shapefile has no CRS. Check that the .prj exists beside: {CORE_AOI_PATH}"
    )

# Work in WGS84 first because Section 6 extracts lat/lon.
core_aoi_wgs84 = core_aoi.to_crs("EPSG:4326").dissolve()
core_aoi_wgs84["geometry"] = core_aoi_wgs84.geometry.buffer(0)

# Cheap prefilter bounds in lon/lat before constructing point geometries.
aoi_min_lon, aoi_min_lat, aoi_max_lon, aoi_max_lat = core_aoi_wgs84.total_bounds

print("Core AOI WGS84 bounds:")
print((aoi_min_lon, aoi_min_lat, aoi_max_lon, aoi_max_lat))


def clip_extracted_df_to_core_aoi(df, level):
    """
    Convert one extracted GEDI granule table to a GeoDataFrame, filter to the
    true core AOI polygon, and return it in raster_crs.
    """

    if df is None or df.empty:
        return gpd.GeoDataFrame(geometry=[], crs=raster_crs)

    required = {"lat", "lon"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"{level}: missing required coordinate columns: {missing}")

    # Drop invalid coordinates.
    df = df.dropna(subset=["lat", "lon"]).copy()

    if df.empty:
        return gpd.GeoDataFrame(geometry=[], crs=raster_crs)

    # Cheap lon/lat bbox prefilter.
    df = df[
        (df["lon"] >= aoi_min_lon) &
        (df["lon"] <= aoi_max_lon) &
        (df["lat"] >= aoi_min_lat) &
        (df["lat"] <= aoi_max_lat)
    ].copy()

    if df.empty:
        return gpd.GeoDataFrame(geometry=[], crs=raster_crs)

    # Create WGS84 point geometries.
    gdf_wgs84 = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326",
    )

    # True AOI polygon filter.
    core_wgs84 = (
        gpd.sjoin(
            gdf_wgs84,
            core_aoi_wgs84[["geometry"]],
            how="inner",
            predicate="within",
        )
        .drop(columns=["index_right"])
        .copy()
    )

    if core_wgs84.empty:
        return gpd.GeoDataFrame(geometry=[], crs=raster_crs)

    # Reproject retained points to raster CRS for downstream raster row/col logic.
    core_native = core_wgs84.to_crs(raster_crs)

    # Add projected coordinate columns. These are useful for later joins/exports.
    core_native["x"] = core_native.geometry.x
    core_native["y"] = core_native.geometry.y

    # Compatibility with earlier trim/inventory code.
    if "source_granule" in core_native.columns and "source_file" not in core_native.columns:
        core_native["source_file"] = core_native["source_granule"]

    return core_native


def extract_product_table_core_aoi(h5_sources, level):
    """
    Extract GEDI footprint records one granule at a time, clip each granule to
    the core AOI, and concatenate only retained AOI footprints.
    """

    kept = []
    failed = []

    n_files = len(h5_sources)
    n_raw_total = 0
    n_core_total = 0
    n_files_with_core = 0

    for h5_source in tqdm(h5_sources, desc=f"Reading and clipping {level} HDF5 granules"):
        try:
            df = extract_gedi_records(h5_source, level=level)

            if df is None or df.empty:
                continue

            n_raw = len(df)
            n_raw_total += n_raw

            core_gdf = clip_extracted_df_to_core_aoi(df, level)
            n_core = len(core_gdf)
            n_core_total += n_core

            if n_core > 0:
                kept.append(core_gdf)
                n_files_with_core += 1

            print(
                f"{level}: {Path(str(h5_source)).name} | "
                f"raw={n_raw:,} | core={n_core:,} | kept_files={n_files_with_core:,}"
            )

            del df, core_gdf
            gc.collect()

        except Exception as exc:
            failed.append((str(h5_source), repr(exc)))
            print(f"WARNING: {level} failed on {Path(str(h5_source)).name}: {repr(exc)}")
            continue

    if failed:
        print()
        print(f"{level}: failed to read {len(failed)} granules")
        for f, e in failed[:10]:
            print(" -", f, e)

    print("=" * 80)
    print(f"{level} core-AOI extraction summary")
    print("=" * 80)
    print(f"HDF5 files scanned:       {n_files:,}")
    print(f"Raw footprints scanned:   {n_raw_total:,}")
    print(f"Core AOI footprints kept: {n_core_total:,}")
    print(f"Files with AOI shots:     {n_files_with_core:,}")
    print(f"Failed files:             {len(failed):,}")
    print("=" * 80)

    if not kept:
        print(f"{level}: no core-AOI footprint records extracted")
        return gpd.GeoDataFrame(geometry=[], crs=raster_crs)

    out = pd.concat(kept, ignore_index=True)
    out = gpd.GeoDataFrame(out, geometry="geometry", crs=raster_crs)

    print(f"{level}: final core-AOI extracted footprints = {len(out):,}")

    return out


# Extract and trim immediately.
l1b_raw = extract_product_table_core_aoi(h5_sources_by_level["L1B"][:2], "L1B")
l2a_raw = extract_product_table_core_aoi(h5_sources_by_level["L2A"][:2], "L2A")

print("L1B columns:", sorted(l1b_raw.columns.tolist()))
print("L2A columns:", sorted(l2a_raw.columns.tolist()))

# Compatibility aliases for downstream notebook sections.
# These are now already CORE-AOI-filtered, not full bbox tables.
l1b_native = l1b_raw
l2a_native = l2a_raw

l1b_core_gdf = l1b_raw
l2a_core_gdf = l2a_raw

print(f"L1B core footprints: {len(l1b_core_gdf):,}")
print(f"L2A core footprints: {len(l2a_core_gdf):,}")

print(type(l1b_raw), len(l1b_raw), l1b_raw.crs)
print(type(l2a_raw), len(l2a_raw), l2a_raw.crs)

display(l1b_raw.head())
display(l2a_raw.head())

In [ ]:
# =============================================================================
# Clip GEDI footprints from raster bbox to the true core AOI shapefile
# =============================================================================
#
# Purpose:
#   The CMR search/download used the raster bounding box, which is rectangular
#   and larger than the ecological AOI. This cell clips extracted GEDI footprints
#   to the true AOI polygon shapefile:
#
#       A:\NCA_DATA\templates\NCA_NAD83UTM11N.shp
#
# Requirements:
#   - l1b_gdf exists as a GeoDataFrame of extracted GEDI01_B footprints
#   - l2a_gdf exists as a GeoDataFrame of extracted GEDI02_A footprints
#   - each GeoDataFrame has a valid geometry column
#   - preferably each GeoDataFrame has a source_file column
#
# Outputs created:
#   - core_aoi
#   - l1b_core_gdf
#   - l2a_core_gdf
#   - used_l1b_files
#   - used_l2a_files
#   - unused_l1b_files
#   - unused_l2a_files
# =============================================================================

from pathlib import Path
import geopandas as gpd

CORE_AOI_PATH = Path(r"A:\NCA_DATA\templates\NCA_NAD83UTM11N.shp")

if not CORE_AOI_PATH.exists():
    raise FileNotFoundError(f"Core AOI shapefile not found: {CORE_AOI_PATH}")

if "l1b_gdf" not in globals():
    raise NameError("l1b_gdf does not exist yet. Run the L1B footprint extraction cell before this cell.")

if "l2a_gdf" not in globals():
    raise NameError("l2a_gdf does not exist yet. Run the L2A footprint extraction cell before this cell.")

if l1b_gdf.empty:
    raise ValueError("l1b_gdf is empty. Cannot clip L1B footprints to AOI.")

if l2a_gdf.empty:
    raise ValueError("l2a_gdf is empty. Cannot clip L2A footprints to AOI.")

if l1b_gdf.crs is None:
    raise ValueError("l1b_gdf has no CRS. Assign or reproject before clipping.")

if l2a_gdf.crs is None:
    raise ValueError("l2a_gdf has no CRS. Assign or reproject before clipping.")

# Read and clean AOI
core_aoi = gpd.read_file(CORE_AOI_PATH)

if core_aoi.empty:
    raise ValueError(f"Core AOI shapefile is empty: {CORE_AOI_PATH}")

if core_aoi.crs is None:
    raise ValueError(
        f"Core AOI shapefile has no CRS. Check that the .prj file exists beside: {CORE_AOI_PATH}"
    )

# Reproject AOI to match GEDI footprint CRS.
# If your GEDI footprint GeoDataFrames are already in EPSG:26911, this will be a no-op.
core_aoi_l1b_crs = core_aoi.to_crs(l1b_gdf.crs)
core_aoi_l2a_crs = core_aoi.to_crs(l2a_gdf.crs)

# Dissolve all AOI features into one geometry.
# This avoids duplicate retained shots if the shapefile has multiple polygons.
core_aoi_l1b_union = core_aoi_l1b_crs.dissolve()
core_aoi_l2a_union = core_aoi_l2a_crs.dissolve()

# Repair invalid geometries if needed.
core_aoi_l1b_union["geometry"] = core_aoi_l1b_union.geometry.buffer(0)
core_aoi_l2a_union["geometry"] = core_aoi_l2a_union.geometry.buffer(0)

# Clip footprints to true AOI polygon.
# predicate="within" keeps footprint points inside the AOI.
l1b_core_gdf = (
    gpd.sjoin(
        l1b_gdf,
        core_aoi_l1b_union[["geometry"]],
        how="inner",
        predicate="within",
    )
    .drop(columns=["index_right"])
    .copy()
)

l2a_core_gdf = (
    gpd.sjoin(
        l2a_gdf,
        core_aoi_l2a_union[["geometry"]],
        how="inner",
        predicate="within",
    )
    .drop(columns=["index_right"])
    .copy()
)

print("=" * 80)
print("GEDI footprint clipping summary")
print("=" * 80)
print(f"Core AOI path: {CORE_AOI_PATH}")
print(f"Core AOI CRS: {core_aoi.crs}")
print()
print(f"L1B bbox footprints:      {len(l1b_gdf):,}")
print(f"L1B core AOI footprints:  {len(l1b_core_gdf):,}")
print(f"L1B retained fraction:    {len(l1b_core_gdf) / len(l1b_gdf):.3f}")
print()
print(f"L2A bbox footprints:      {len(l2a_gdf):,}")
print(f"L2A core AOI footprints:  {len(l2a_core_gdf):,}")
print(f"L2A retained fraction:    {len(l2a_core_gdf) / len(l2a_gdf):.3f}")

# Track which raw HDF5 files actually contributed at least one core-AOI shot.
# This assumes your footprint extractor wrote a source_file column.
if "source_file" in l1b_core_gdf.columns:
    used_l1b_files = set(l1b_core_gdf["source_file"].dropna().unique())
    print()
    print(f"L1B files with >=1 core AOI footprint: {len(used_l1b_files):,}")
else:
    used_l1b_files = set()
    print()
    print("WARNING: l1b_core_gdf has no 'source_file' column. Cannot identify used L1B files.")

if "source_file" in l2a_core_gdf.columns:
    used_l2a_files = set(l2a_core_gdf["source_file"].dropna().unique())
    print(f"L2A files with >=1 core AOI footprint: {len(used_l2a_files):,}")
else:
    used_l2a_files = set()
    print("WARNING: l2a_core_gdf has no 'source_file' column. Cannot identify used L2A files.")

# Optional inventory of unused raw files.
# This does not delete or move anything. It only reports candidates.
l1b_raw_dir = Path(r"C:\NCA_DATA\NCA_template_bbox_L1B_L2A\raw_h5\GEDI01_B")
l2a_raw_dir = Path(r"C:\NCA_DATA\NCA_template_bbox_L1B_L2A\raw_h5\GEDI02_A")

if l1b_raw_dir.exists() and used_l1b_files:
    all_l1b_files = {p.name for p in l1b_raw_dir.glob("*.h5")}
    unused_l1b_files = all_l1b_files - used_l1b_files
    print()
    print(f"All downloaded L1B .h5 files:          {len(all_l1b_files):,}")
    print(f"Unused L1B files outside core AOI:     {len(unused_l1b_files):,}")
else:
    unused_l1b_files = set()

if l2a_raw_dir.exists() and used_l2a_files:
    all_l2a_files = {p.name for p in l2a_raw_dir.glob("*.h5")}
    unused_l2a_files = all_l2a_files - used_l2a_files
    print(f"All downloaded L2A .h5 files:          {len(all_l2a_files):,}")
    print(f"Unused L2A files outside core AOI:     {len(unused_l2a_files):,}")
else:
    unused_l2a_files = set()

print("=" * 80)

# From this point forward, use:
#   l1b_core_gdf instead of l1b_gdf
#   l2a_core_gdf instead of l2a_gdf
#
# Example downstream join:
#
# joined_core_gdf = l1b_core_gdf.merge(
#     l2a_core_gdf.drop(columns="geometry"),
#     on="shot_number",
#     how="inner",
#     suffixes=("_l1b", "_l2a")
# )
#
# joined_core_gdf = gpd.GeoDataFrame(
#     joined_core_gdf,
#     geometry="geometry",
#     crs=l1b_core_gdf.crs,
# )
#
# print(f"Joined core AOI L1B/L2A shots: {len(joined_core_gdf):,}")

## 9. Join L1B and L2A by shot number

`shot_number` is the main cross-product key. This joined table keeps both L1B waveform-index fields and L2A interpreted metrics. Geometry is taken from L2A when available.


In [ ]:
# =============================================================================
# 9. Join core-AOI-filtered GEDI L1B and L2A footprints
# =============================================================================
#
# This section assumes the AOI clipping cell has already been run and produced:
#
#   l1b_core_gdf
#   l2a_core_gdf
#
# The joined product uses L2A geometry because L2A carries the RH/elevation/
# quality fields and is the cleaner footprint-level table for mapping.
# L1B waveform metadata are joined as attributes.
# =============================================================================

import geopandas as gpd
import pandas as pd


def join_l1b_l2a_core(l1b_core_gdf, l2a_core_gdf, raster_crs=None, use_beam_key=True):
    """
    Join AOI-filtered GEDI L1B and L2A footprints by shot_number.

    Parameters
    ----------
    l1b_core_gdf : geopandas.GeoDataFrame
        AOI-filtered GEDI01_B / L1B footprints.

    l2a_core_gdf : geopandas.GeoDataFrame
        AOI-filtered GEDI02_A / L2A footprints.

    raster_crs : CRS-like, optional
        Fallback CRS if the input GeoDataFrames lack CRS.

    use_beam_key : bool
        If True, join by shot_number + beam when beam exists in both tables.
        If False, join by shot_number only.

    Returns
    -------
    geopandas.GeoDataFrame
        Joined core-AOI L1B/L2A footprints.
    """

    output_crs = l2a_core_gdf.crs if l2a_core_gdf.crs is not None else raster_crs

    if l1b_core_gdf.empty or l2a_core_gdf.empty:
        print("Cannot join: one product has zero core-AOI-filtered footprints.")
        print(f"L1B core footprints: {len(l1b_core_gdf):,}")
        print(f"L2A core footprints: {len(l2a_core_gdf):,}")
        return gpd.GeoDataFrame(geometry=[], crs=output_crs)

    if "shot_number" not in l1b_core_gdf.columns:
        raise KeyError("Cannot join: 'shot_number' missing from L1B core table.")

    if "shot_number" not in l2a_core_gdf.columns:
        raise KeyError("Cannot join: 'shot_number' missing from L2A core table.")

    # Work on copies so the originals are not modified.
    l1b_work = l1b_core_gdf.copy()
    l2a_work = l2a_core_gdf.copy()

    # Normalize shot_number to string for robust joining.
    # This protects against int/string dtype mismatches.
    l1b_work["_shot_join"] = l1b_work["shot_number"].astype(str)
    l2a_work["_shot_join"] = l2a_work["shot_number"].astype(str)

    join_keys = ["_shot_join"]

    # Optional conservative beam join.
    # If this gives zero matches, we retry below with shot_number only.
    if use_beam_key and "beam" in l1b_work.columns and "beam" in l2a_work.columns:
        l1b_work["_beam_join"] = l1b_work["beam"].astype(str)
        l2a_work["_beam_join"] = l2a_work["beam"].astype(str)
        join_keys.append("_beam_join")

    # Preserve L2A geometry. Drop L1B geometry before merge.
    l1b_attrs = l1b_work.drop(columns=[l1b_work.geometry.name], errors="ignore").copy()

    joined = l2a_work.merge(
        l1b_attrs,
        on=join_keys,
        how="inner",
        suffixes=("_l2a", "_l1b"),
    )

    print("=" * 80)
    print("Core AOI GEDI L1B/L2A join summary")
    print("=" * 80)
    print(f"L1B core footprints:        {len(l1b_core_gdf):,}")
    print(f"L2A core footprints:        {len(l2a_core_gdf):,}")
    print(f"Joined L1B/L2A footprints:  {len(joined):,}")
    print(f"Join keys used:             {join_keys}")

    if len(l1b_core_gdf) > 0:
        print(f"L1B retained in join:       {len(joined) / len(l1b_core_gdf):.3f}")

    if len(l2a_core_gdf) > 0:
        print(f"L2A retained in join:       {len(joined) / len(l2a_core_gdf):.3f}")

    print("=" * 80)

    if joined.empty:
        return gpd.GeoDataFrame(joined, geometry=[], crs=output_crs)

    joined_core_gdf = gpd.GeoDataFrame(
        joined,
        geometry=l2a_work.geometry.name,
        crs=output_crs,
    )

    # Remove helper join columns.
    helper_cols = [c for c in ["_shot_join", "_beam_join"] if c in joined_core_gdf.columns]
    joined_core_gdf = joined_core_gdf.drop(columns=helper_cols)

    # Restore a single canonical shot_number column.
    if "shot_number_l2a" in joined_core_gdf.columns:
        joined_core_gdf["shot_number"] = joined_core_gdf["shot_number_l2a"]
    elif "shot_number_l1b" in joined_core_gdf.columns:
        joined_core_gdf["shot_number"] = joined_core_gdf["shot_number_l1b"]

    # Put the most important columns first when present.
    front_cols = [
        c for c in [
            "shot_number",
            "beam_l2a",
            "beam_l1b",
            "source_file_l2a",
            "source_file_l1b",
            "quality_flag",
            "degrade_flag",
            "sensitivity",
            "rh98",
            "rh100",
        ]
        if c in joined_core_gdf.columns
    ]

    geometry_col = joined_core_gdf.geometry.name
    remaining_cols = [
        c for c in joined_core_gdf.columns
        if c not in front_cols and c != geometry_col
    ]

    joined_core_gdf = joined_core_gdf[front_cols + remaining_cols + [geometry_col]]

    return joined_core_gdf


# Primary conservative join: shot_number + beam if beam exists in both.
joined_core_gdf = join_l1b_l2a_core(
    l1b_core_gdf,
    l2a_core_gdf,
    raster_crs=raster_crs,
    use_beam_key=True,
)

# Fallback: if beam labels differ between L1B/L2A and the conservative join fails,
# retry using shot_number only.
if joined_core_gdf.empty:
    print("\nBeam-key join returned zero rows. Retrying with shot_number only...\n")

    joined_core_gdf = join_l1b_l2a_core(
        l1b_core_gdf,
        l2a_core_gdf,
        raster_crs=raster_crs,
        use_beam_key=False,
    )

# Compatibility aliases for later notebook sections that still expect *_native names.
# From this point forward, these names refer to CORE-AOI-filtered footprints.
l1b_native = l1b_core_gdf
l2a_native = l2a_core_gdf
joined_native = joined_core_gdf

display(joined_core_gdf.head())
print(f"Final joined_core_gdf rows: {len(joined_core_gdf):,}")

## 10. Optional L2A quality filtering

This creates a conservative L2A subset for science analyses while preserving the unfiltered L2A output. Adjust this deliberately; dryland/low-stature vegetation work should inspect the effect of these filters before making them final.


In [ ]:
l2a_quality = l2a_native.copy()

if not l2a_quality.empty:
    if "quality_flag" in l2a_quality.columns:
        l2a_quality = l2a_quality[l2a_quality["quality_flag"] == 1].copy()

    if "degrade_flag" in l2a_quality.columns:
        l2a_quality = l2a_quality[l2a_quality["degrade_flag"] == 0].copy()

print(f"L2A quality-filtered footprints: {len(l2a_quality):,}")
l2a_quality.head()


## 11. Helper: extract one L1B receive waveform by shot number

Use this only after identifying shots of interest. The exact waveform dataset names can vary, so this helper tries common L1B dataset locations. It returns the receive waveform segment for a single shot when `rxwaveform`, `rx_sample_start_index`, and `rx_sample_count` are available.


In [ ]:
def extract_l1b_rx_waveform_for_shot(h5_source, shot_number, beam_name=None):
    '''
    Return a single L1B receive waveform segment for a shot_number.

    Parameters
    ----------
    h5_source : path-like or file-like
        Local HDF5 path or earthaccess-opened file handle.
    shot_number : str or int
        GEDI shot number to extract.
    beam_name : str, optional
        Beam to search first, e.g. 'BEAM0110'. If None, search all beams.

    Returns
    -------
    dict with source, beam, shot_number, rxwaveform, rx_sample_start_index, rx_sample_count
    '''
    target = str(shot_number)
    with h5py.File(h5_source, "r") as h5:
        beams = [beam_name] if beam_name else list_beams(h5)
        for beam in beams:
            if beam not in h5:
                continue
            g = h5[beam]
            shots, _ = read_optional_1d(g, ["shot_number", "geolocation/shot_number"], n=None)
            if shots is None:
                continue
            shots = shots.astype(str)
            idx = np.where(shots == target)[0]
            if len(idx) == 0:
                continue
            i = int(idx[0])

            starts, _ = read_optional_1d(g, ["rx_sample_start_index"], n=len(shots))
            counts, _ = read_optional_1d(g, ["rx_sample_count"], n=len(shots))
            if starts is None or counts is None:
                raise KeyError("rx_sample_start_index or rx_sample_count not found for this beam.")

            rx_arr, rx_path = read_first_dataset(g, ["rxwaveform", "rx_waveform"])
            if rx_arr is None:
                raise KeyError("rxwaveform dataset not found for this beam.")

            # GEDI sample start indices are commonly 1-based; this conversion is safe for standard L1B usage.
            start = int(starts[i]) - 1
            count = int(counts[i])
            waveform = np.asarray(rx_arr[start:start + count])

            return {
                "source_granule": infer_granule_name(h5_source),
                "beam": beam,
                "shot_number": target,
                "rxwaveform_path": rx_path,
                "rx_sample_start_index": int(starts[i]),
                "rx_sample_count": count,
                "rxwaveform": waveform,
            }

    raise ValueError(f"Shot number {shot_number} not found in {infer_granule_name(h5_source)}")

# Example use after picking a shot from l1b_native:
# example_shot = l1b_native.iloc[0]["shot_number"]
# example_granule = RAW_DIR / GEDI_PRODUCTS["L1B"] / l1b_native.iloc[0]["source_granule"]
# wave = extract_l1b_rx_waveform_for_shot(example_granule, example_shot, beam_name=l1b_native.iloc[0]["beam"])
# wave["rxwaveform"][:10]


## 12. Save outputs


In [ ]:
# GPKG layers
if not l1b_native.empty:
    l1b_native.to_file(OUT_GPKG, layer="GEDI01_B_L1B", driver="GPKG")
if not l2a_native.empty:
    l2a_native.to_file(OUT_GPKG, layer="GEDI02_A_L2A", driver="GPKG")
if not joined_native.empty:
    joined_native.to_file(OUT_GPKG, layer="L1B_L2A_joined_by_shot_number", driver="GPKG")
if not l2a_quality.empty:
    l2a_quality.to_file(OUT_GPKG, layer="GEDI02_A_L2A_quality_filtered", driver="GPKG")

# CSVs
if not l1b_native.empty:
    l1b_native.drop(columns="geometry").to_csv(OUT_L1B_CSV, index=False)
if not l2a_native.empty:
    l2a_native.drop(columns="geometry").to_csv(OUT_L2A_CSV, index=False)
if not joined_native.empty:
    joined_native.drop(columns="geometry").to_csv(OUT_JOINED_CSV, index=False)

print("Saved outputs:")
print(" -", OUT_GPKG)
print(" -", OUT_L1B_CSV)
print(" -", OUT_L2A_CSV)
print(" -", OUT_JOINED_CSV)


## 13. Quick map check


In [ ]:
plot_l1b = l1b_native.sample(min(len(l1b_native), 5000), random_state=42) if len(l1b_native) else l1b_native
plot_l2a = l2a_native.sample(min(len(l2a_native), 5000), random_state=42) if len(l2a_native) else l2a_native

ax = aoi_native.boundary.plot(figsize=(7, 7), linewidth=2)
if len(plot_l1b):
    plot_l1b.plot(ax=ax, markersize=1, alpha=0.5)
if len(plot_l2a):
    plot_l2a.plot(ax=ax, markersize=1, alpha=0.5)
ax.set_title("GEDI L1B and L2A footprints inside raster bbox")
ax.set_xlabel(f"X ({raster_crs})")
ax.set_ylabel(f"Y ({raster_crs})")


## Notes for adapting this notebook

- `GEDI01_B` is the L1B geolocated waveform product. Use it when you need waveform-level information, receive-waveform sample indices, and waveform extraction.
- `GEDI02_A` is the L2A elevation and relative-height product. Use it when you need interpreted ground/canopy metrics such as `rh95`, `rh98`, `rh100`, `elev_lowestmode`, and quality flags.
- The spatial query is bounding-box based at CMR search time; exact footprint filtering is performed after reading the HDF5 files.
- For science use, inspect `quality_flag`, `degrade_flag`, `sensitivity`, beam type, slope/incidence geometry, and acquisition dates before final filtering.
- For dryland/low-stature systems, do not blindly drop ambiguous L2A returns without checking whether filtering preferentially removes the exact sparse/low-canopy cases you care about.
